#### Sprint 04 — Fine-tuning do modelo pysentimiento
###### **Alunos**: Maria Eduarda da Cruz | Yghor Kristian Andrade | **Orientador:** Rômulo Francisco de Souza
###### **Base**: df_posts.csv (750 posts, domínio político-universitário)
###### **Modelo base**: pysentimiento RoBERTuito PT-BR

##### 01. Importando Bibliotecas e Instalações

In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
import numpy as np
import os

# Verifica dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
# Carrega a base gerada
df = pd.read_csv("opinate_data/df_posts.csv")
print(f"\n==== Base carregada: {len(df):,} posts ====")
print(df["sentimento"].value_counts().to_string())

c:\Users\Fenec\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cpu
PyTorch: 2.12.0+cpu

==== Base carregada: 725 posts ====
sentimento
neutro      261
negativo    244
positivo    220


##### 02. Preparação dos Dados

In [2]:
LABEL2ID = {"negativo": 0, "neutro": 1, "positivo": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df["label"] = df["sentimento"].map(LABEL2ID)
df = df[["conteudo", "label"]].dropna().reset_index(drop=True)

# Split: 70% treino, 15% validação, 15% teste
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(f"Treino:    {len(train_df)} posts")
print(f"Validação: {len(val_df)} posts")
print(f"Teste:     {len(test_df)} posts")
print(f"\nDistribuição treino:")
print(train_df["label"].map(ID2LABEL).value_counts().to_string())

Treino:    507 posts
Validação: 109 posts
Teste:     109 posts

Distribuição treino:
label
neutro      182
negativo    171
positivo    154


##### 03. Tokenizadores e Dataset

In [3]:
MODEL_NAME = "pysentimiento/robertuito-sentiment-analysis"

print(f"==== Carregando tokenizer: {MODEL_NAME} ====")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("==== Tokenizer carregado! ====")

class OpinateDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_len=128):
        self.textos    = textos
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.textos[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = OpinateDataset(train_df["conteudo"].tolist(), train_df["label"].tolist(), tokenizer)
val_dataset   = OpinateDataset(val_df["conteudo"].tolist(),   val_df["label"].tolist(),   tokenizer)
test_dataset  = OpinateDataset(test_df["conteudo"].tolist(),  test_df["label"].tolist(),  tokenizer)

print(f"\n==== Datasets criados! ====")
print(f"   Exemplo de encoding: {train_dataset[0]['input_ids'].shape}")

==== Carregando tokenizer: pysentimiento/robertuito-sentiment-analysis ====


c:\Users\Fenec\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Fenec\.cache\huggingface\hub\models--pysentimiento--robertuito-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


==== Tokenizer carregado! ====

==== Datasets criados! ====
   Exemplo de encoding: torch.Size([128])


##### 04. Modelo e Configuração de Treino

In [5]:
print(f"==== Carregando modelo base: {MODEL_NAME} ====")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
print("==== Modelo carregado! ====")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1_macro":  f1_score(labels, preds, average="macro"),
        "f1_neg":    f1_score(labels, preds, labels=[0], average="macro"),
        "f1_neu":    f1_score(labels, preds, labels=[1], average="macro"),
        "f1_pos":    f1_score(labels, preds, labels=[2], average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./modelo/pysentimiento_finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=8,       # pequeno pra CPU não travar
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    save_total_limit=2,
    report_to="none",                    # desliga wandb
    use_cpu=True,                        # força CPU explicitamente
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n==== Trainer configurado! ====")
print(f"   Épocas:      {training_args.num_train_epochs}")
print(f"   Batch size:  {training_args.per_device_train_batch_size}")
print(f"   LR:          {training_args.learning_rate}")
print(f"   Early stop:  paciência de 2 épocas sem melhora")

==== Carregando modelo base: pysentimiento/robertuito-sentiment-analysis ====


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2965.48it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


==== Modelo carregado! ====

==== Trainer configurado! ====
   Épocas:      5
   Batch size:  8
   LR:          2e-05
   Early stop:  paciência de 2 épocas sem melhora


##### 05. Treinamento do Modelo (sem GPU)

In [6]:
print("==== Iniciando fine-tuning... ====")
print("~~~~ Pode demorar 20-40 minutos em CPU. ~~~~")
print("---- Acompanhe o F1-Macro a cada época.\n ----")

trainer.train()

print("\n==== Fine-tuning concluído! ✔️ ====")
print(f"==== Melhor F1-Macro na validação: {trainer.state.best_metric:.4f} ====")

==== Iniciando fine-tuning... ====
~~~~ Pode demorar 20-40 minutos em CPU. ~~~~
---- Acompanhe o F1-Macro a cada época.
 ----


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Neg,F1 Neu,F1 Pos
1,0.672029,0.196243,0.990306,0.986301,1.000000,0.984615
2,0.130938,0.007444,1.000000,1.000000,1.000000,1.000000
3,0.001805,0.000918,1.000000,1.000000,1.000000,1.000000
4,0.001335,0.000638,1.000000,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.96s/it]



==== Fine-tuning concluído! ✔️ ====
==== Melhor F1-Macro na validação: 1.0000 ====


##### 06. Avaliação do Teste + Saving do Modelo

In [7]:
# Salva modelo fine-tuned
os.makedirs("modelo/pysentimiento_finetuned", exist_ok=True)
trainer.save_model("modelo/pysentimiento_finetuned")
tokenizer.save_pretrained("modelo/pysentimiento_finetuned")
print("✅ Modelo salvo em modelo/pysentimiento_finetuned/")

# Avalia no conjunto de teste
print("\n==== Avaliando no conjunto de teste... ====")
preds_output = trainer.predict(test_dataset)
preds        = np.argmax(preds_output.predictions, axis=1)
labels_true  = preds_output.label_ids

print(f"\n{'='*50}")
print("==== RESULTADO FINE-TUNED — Conjunto de Teste ====")
print(f"{'='*50}")
print(classification_report(labels_true, preds,
      target_names=["negativo", "neutro", "positivo"], digits=3))
print(f"F1-Macro: {f1_score(labels_true, preds, average='macro'):.4f}")

print(f"\n{'='*50}")
print("==== COMPARATIVO ZERO-SHOT vs. FINE-TUNED ====")
print(f"{'='*50}")
print(f"  Zero-shot (TweetSentBR):       F1-Macro 0.8897")
print(f"  Zero-shot (Base Universitária): F1-Macro 0.5030")
print(f"  Fine-tuned (Base Universitária): F1-Macro {f1_score(labels_true, preds, average='macro'):.4f}")

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.00s/it]


✅ Modelo salvo em modelo/pysentimiento_finetuned/

==== Avaliando no conjunto de teste... ====



==== RESULTADO FINE-TUNED — Conjunto de Teste ====
              precision    recall  f1-score   support

    negativo      1.000     1.000     1.000        37
      neutro      1.000     1.000     1.000        39
    positivo      1.000     1.000     1.000        33

    accuracy                          1.000       109
   macro avg      1.000     1.000     1.000       109
weighted avg      1.000     1.000     1.000       109

F1-Macro: 1.0000

==== COMPARATIVO ZERO-SHOT vs. FINE-TUNED ====
  Zero-shot (TweetSentBR):       F1-Macro 0.8897
  Zero-shot (Base Universitária): F1-Macro 0.5030
  Fine-tuned (Base Universitária): F1-Macro 1.0000
